In [64]:

# ==============================
# Wage Prediction Model Notebook
# ==============================

# 1. Import Libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score


In [65]:
# 2. Load Data

file_path = "../data/wages_by_education_cleaned_v2.xlsx" # Replace with your file path

# 📥 Load the Excel file
try:
    # specify engine to ensure pandas uses openpyxl
    df = pd.read_excel(file_path, engine='openpyxl')

    print("File loaded successfully!")
    print(df.head())
    # Normalize EducationLevel labels
    df["EducationLevel"] = df["EducationLevel"].str.replace(" Degree", "", regex=False).str.title()
    # Show first few rows
    print(df.head())
    print(df.columns)  # Show column names
except FileNotFoundError: # Gives a error message if file path is incorrect
    print("File not found. Double-check the path.")
except Exception as e:
    print(f"An error occurred: {e}")


File loaded successfully!
   EmployeeID  EmployeeAge EmployeeGender    CareerField CareerLevel  \
0           1           57         Female  Manufacturing       Entry   
1           2           57           Male    Engineering         Mid   
2           3           55           Male   Construction       Entry   
3           4           57           Male          Media       Entry   
4           5           57           Male    Real Estate         Mid   

   MonthlyIncome     EducationLevel  
0           4288   Associate Degree  
1           7090  Bachelor's Degree  
2           4734   Associate Degree  
3           7082    Master's Degree  
4           4753        High School  
   EmployeeID  EmployeeAge EmployeeGender    CareerField CareerLevel  \
0           1           57         Female  Manufacturing       Entry   
1           2           57           Male    Engineering         Mid   
2           3           55           Male   Construction       Entry   
3           4           5

In [66]:

# 3. Encode Categorical Features
career_encoder = LabelEncoder()
education_encoder = LabelEncoder()

df['CareerField_encoded'] = career_encoder.fit_transform(df['CareerField'])
df['EducationLevel_encoded'] = education_encoder.fit_transform(df['EducationLevel'])

# 4. Prepare Features and Target
X = df[['CareerField_encoded', 'EducationLevel_encoded']]
y = df['MonthlyIncome']


In [67]:
# 5. Split Data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [68]:
# 6. Initialize Models
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42)
}


In [69]:
# 7. Train and Evaluate Models
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    mse = mean_squared_error(y_test, predictions)
    r2 = r2_score(y_test, predictions) # R-squared score
    results.append((name, mse, r2))
    print(f"{name} - MSE: {mse:.2f}, R²: {r2:.4f}")

Linear Regression - MSE: 2247609.54, R²: 0.3102
Decision Tree - MSE: 540208.02, R²: 0.8342
Random Forest - MSE: 539836.97, R²: 0.8343
Gradient Boosting - MSE: 536573.33, R²: 0.8353


In [70]:
import joblib
import os

# 8. Select Best Model (based on R²)
best_model_name, best_mse, best_r2 = max(results, key=lambda x: x[2])
best_model = models[best_model_name]
print(f"\n✅ Best Model: {best_model_name} (R² = {best_r2:.4f})")

# Ensure models folder exists outside notebooks
save_dir = "../models"
os.makedirs(save_dir, exist_ok=True)

# Save model and encoders together
save_path = os.path.join(save_dir, f"{best_model_name}_bundle.pkl")
joblib.dump({
    "model": best_model,
    "career_encoder": career_encoder,
    "education_encoder": education_encoder
}, save_path)

print(f"📁 Model and encoders saved as {save_path}")


✅ Best Model: Gradient Boosting (R² = 0.8353)
📁 Model and encoders saved as ../models\Gradient Boosting_bundle.pkl
